# Intent Model Optimization & Evaluation

This notebook tests classifier architectures, tunes hyperparameters, evaluates sub-category confusion, and validates the escalation engine integration.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath('..'))
from src.classifier import IntentClassifier
from src.escalation import EscalationEngine
from src.reply_generator import ReplyGenerator


## 1. Load Annotated Golden Dataset

In [2]:
df = pd.read_csv('../data/golden_set.csv')
df.sample(5, random_state=42)

## 2. End-to-End System Evaluation (Intent + Escalation)

In [3]:
classifier = IntentClassifier()
escalation_engine = EscalationEngine()
reply_generator = ReplyGenerator()

results = []
for _, row in df.iterrows():
    text = row['text']
    pred = classifier.predict_one(text)
    esc = escalation_engine.evaluate(text, pred['intent'])
    reply = reply_generator.generate_reply(text, pred['intent'], esc)
    
    results.append({
        'text': text,
        'true_intent': row['intent'],
        'pred_intent': pred['intent'],
        'confidence': pred['confidence'],
        'true_escalate': row['escalate'],
        'pred_escalate': esc['escalate'],
        'urgency': esc['urgency'],
        'reply_preview': reply[:75] + '...'
    })

res_df = pd.DataFrame(results)
res_df.head(5)

## 3. Confusion Matrix Visualization

In [4]:
from sklearn.metrics import confusion_matrix

labels = sorted(df['intent'].unique())
cm = confusion_matrix(res_df['true_intent'], res_df['pred_intent'], labels=labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel('Predicted Intent')
plt.ylabel('True Intent')
plt.title('Apple Support Intent Classification Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../results/confusion_matrix.png', dpi=300)
plt.show()

## 4. Escalation Policy Breakdown

In [5]:
print("Escalation Distribution:")
print(res_df['pred_escalate'].value_counts())

print("\nUrgency Distribution:")
print(res_df['urgency'].value_counts())